# Enhanced Whisper Pipeline - DATA-CENTRIC APPROACH

Maintains your original analysis & selection workflow while adding:
- VAD (Pyannote 3.0) for voice activity detection
- Smart merging to enforce 30+ second chunks
- CHA transcript alignment with audio cuts

## 6-Zone Architecture

1. **ZONE 1: File Matching** - Match CHA ↔ Audio
2. **ZONE 2: Extract & Analyze** - Word-level segments + visualization
3. **ZONE 3: Segment Audio** - VAD + intelligent merging (NEW)
4. **ZONE 4: Speaker Info** - Extract speaker metadata
5. **ZONE 5: Evaluate Whisper** - Calculate WER
6. **ZONE 6: Create Dataset** - Train/eval split

## Installation & Setup

In [ ]:
!pip install -q pyannote.audio openai-whisper librosa soundfile jiwer pandas numpy
!apt-get install -y ffmpeg libsndfile1 > /dev/null 2>&1

print("✓ Dependencies installed")

## Imports & Configuration

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple
from datetime import datetime
import logging

import librosa
import soundfile as sf
import whisper
from jiwer import wer as compute_wer
from tqdm import tqdm

# Configure
logging.basicConfig(level=logging.INFO)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

CONFIG = {
    "cha_dir": Path("/content/drive/MyDrive/asr/data/cha/1"),
    "audio_dir": Path("/content/drive/MyDrive/asr/data/songs/1"),
    "output_dir": Path("/content/drive/MyDrive/asr/output/whisper_children_dataset"),
    "whisper_model": "base",
    "sample_size": None,
    "train_ratio": 0.8,
    "sample_rate": 16000,
    "audio_extensions": [".wav", ".mp3", ".m4a", ".flac"],
    # NEW: Enhancement parameters
    "min_chunk_duration": 30.0,
    "use_vad": True,
    "save_plots": True,
}

CHILD_ROLES = {"Target_Child", "Child", "Sibling", "Peer", "Playmate"}
ADULT_ROLES = {"Investigator", "Teacher", "Mother", "Father", "Adult", "Caregiver", "Parent"}

print("✓ Configuration loaded")

## Data Structures

In [ ]:
@dataclass
class WordSegment:
    """Word-level segment from CHA file"""
    speaker: str
    text: str
    words: list  # [(word, start_ms, end_ms)]
    file_name: str = ""

@dataclass
class AudioChunk:
    """Audio chunk after merging to meet minimum duration"""
    file_name: str
    segment_number: int
    audio_data: np.ndarray
    sample_rate: int
    start_time: float
    end_time: float
    duration: float
    utterances: List
    speech_ratio: float = 0.0
    vad_segments: List = None

print("✓ Data structures defined")

# ZONE 1: FILE MATCHING

In [ ]:
def find_matching_files(cha_dir: Path, audio_dir: Path, audio_extensions: List[str]) -> Dict:
    """Match CHA and audio files"""
    
    cha_files = sorted(set(list(cha_dir.glob("*.cha")) + list(cha_dir.glob("**/*.cha"))))
    
    matched = []
    unmatched_cha = []
    
    for cha_file in cha_files:
        base_name = cha_file.stem
        audio_file = None
        
        for ext in audio_extensions:
            potential = audio_dir / f"{base_name}{ext}"
            if potential.exists():
                audio_file = potential
                break
        
        if audio_file:
            matched.append((cha_file, audio_file))
            print(f"✓ {cha_file.name:30s} ↔ {audio_file.name}")
        else:
            unmatched_cha.append(cha_file.name)
    
    return {"matched": matched, "unmatched_cha": unmatched_cha, "count": len(matched)}

print("\n" + "="*70)
print("ZONE 1: FILE MATCHING")
print("="*70)

match_result = find_matching_files(
    CONFIG["cha_dir"],
    CONFIG["audio_dir"],
    CONFIG["audio_extensions"]
)

print(f"\nMatched: {match_result['count']} pairs")
print(f"Unmatched CHA: {len(match_result['unmatched_cha'])}")
print("="*70)

# ZONE 2: EXTRACT & ANALYZE SEGMENTS

In [ ]:
def _extract_from_file(cha_file: Path) -> List[WordSegment]:
    """Extract word-level segments from CHA file"""
    
    segments = []
    current_speaker = None
    file_name = cha_file.stem

    with cha_file.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()

            if line.startswith("*"):
                current_speaker = line.split(":", 1)[0].replace("*", "").strip()

            elif line.startswith("%wor:"):
                if not current_speaker:
                    continue

                wor_content = line.split(":", 1)[1].strip()
                wor_content = wor_content.replace('\x15', '')

                tokens = wor_content.split()
                words = []
                i = 0
                
                while i < len(tokens):
                    token = tokens[i]

                    if re.match(r"^\d{5,}_\d{5,}$", token):
                        if words:
                            word, _, _ = words[-1]
                            match = re.match(r"(\d+)_(\d+)", token)
                            if match:
                                start, end = int(match.group(1)), int(match.group(2))
                                words[-1] = (word, start, end)
                        i += 1
                        continue

                    words.append((token, None, None))
                    i += 1

                words_with_ts = [(w, s, e) for w, s, e in words if s is not None and e is not None]

                if not words_with_ts:
                    continue

                clean_words = [w for w, _, _ in words_with_ts if w not in ('?', '.', ',', '!', '+...')]

                if clean_words:
                    clean_text = " ".join(clean_words)
                    segments.append(
                        WordSegment(
                            speaker=current_speaker,
                            text=clean_text,
                            words=words_with_ts,
                            file_name=file_name
                        )
                    )

    return segments

def extract_segments_from_matched(matched_pairs: List[Tuple[Path, Path]]) -> List[WordSegment]:
    """Extract all segments"""
    
    print("\n" + "="*70)
    print("ZONE 2: EXTRACT & ANALYZE SEGMENTS")
    print("="*70)
    
    all_segments = []
    
    for cha_file, _ in matched_pairs:
        segments = _extract_from_file(cha_file)
        all_segments.extend(segments)
        print(f"  {cha_file.name:30s}: {len(segments):4d} segments")
    
    print(f"\n  Total: {len(all_segments)} segments")
    print("="*70)
    
    return all_segments

# Extract
segments = extract_segments_from_matched(match_result["matched"])

## DATA ANALYSIS FOR SELECTION

In [ ]:
# Build analysis dataframe
analysis_data = []

for segment in segments:
    if segment.words:
        start_ms = min(w[1] for w in segment.words)
        end_ms = max(w[2] for w in segment.words)
        duration = (end_ms - start_ms) / 1000.0
    else:
        duration = 0
    
    word_count = len(segment.text.split())
    
    analysis_data.append({
        'file_name': segment.file_name,
        'speaker': segment.speaker,
        'text': segment.text,
        'duration': duration,
        'word_count': word_count,
        'words_per_second': word_count / duration if duration > 0 else 0
    })

df_analysis = pd.DataFrame(analysis_data)
print("\n" + "="*70)
print("SEGMENT STATISTICS")
print("="*70)
print(f"\nDuration (seconds):")
print(df_analysis['duration'].describe())
print(f"\nWord count:")
print(df_analysis['word_count'].describe())
print(f"\nSpeaker distribution:")
print(df_analysis['speaker'].value_counts())
print("="*70)

## VISUALIZATION: Data-Driven Selection

In [ ]:
# Duration distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Duration histogram
axes[0, 0].hist(df_analysis['duration'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(CONFIG['min_chunk_duration'], color='red', linestyle='--', label='Min chunk (30s)')
axes[0, 0].set_xlabel('Duration (seconds)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Segment Duration Distribution')
axes[0, 0].legend()

# Speaker distribution
df_analysis['speaker'].value_counts().plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Speaker Distribution')
axes[0, 1].set_ylabel('Count')

# Word count vs duration
axes[1, 0].scatter(df_analysis['duration'], df_analysis['word_count'], alpha=0.5)
axes[1, 0].set_xlabel('Duration (seconds)')
axes[1, 0].set_ylabel('Word Count')
axes[1, 0].set_title('Word Count vs Duration')

# Words per second by speaker
df_analysis.boxplot(column='words_per_second', by='speaker', ax=axes[1, 1])
axes[1, 1].set_title('Speaking Rate by Speaker')
axes[1, 1].set_ylabel('Words/Second')

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'analysis_plots.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Saved analysis plots")

## DATA-DRIVEN SEGMENT SELECTION

Based on analysis, select segments. Examples:

```python
# High quality segments (long, many words)
good_segments = df_analysis[
    (df_analysis['duration'] >= 2) &
    (df_analysis['word_count'] >= 5)
]

# By speaker
child_segments = df_analysis[df_analysis['speaker'] == 'CHI']

# By file
file_segments = df_analysis[df_analysis['file_name'] == '01-1a']
```

In [ ]:
# EXAMPLE: Select segments based on criteria
# You can modify these criteria based on your analysis above

selected_df = df_analysis[
    (df_analysis['duration'] >= 0.5) &  # At least 0.5 seconds
    (df_analysis['word_count'] >= 2)     # At least 2 words
]

print(f"\nSelection summary:")
print(f"  Total segments: {len(df_analysis)}")
print(f"  Selected: {len(selected_df)} ({len(selected_df)/len(df_analysis)*100:.1f}%)")
print(f"  Removed: {len(df_analysis) - len(selected_df)}")

# Filter original segments list
selected_text_set = set(zip(selected_df['file_name'], selected_df['text']))
selected_segments = [s for s in segments if (s.file_name, s.text) in selected_text_set]

print(f"\nSelected segments for processing: {len(selected_segments)}")

# ZONE 3: SEGMENT AUDIO WITH SMART MERGING (ENHANCED)

In [ ]:
# VAD Processor
class VADProcessor:
    def __init__(self):
        try:
            from pyannote.audio import Pipeline
            import torch
            self.pipeline = Pipeline.from_pretrained("pyannote/segmentation-3.0")
            if torch.cuda.is_available():
                self.pipeline = self.pipeline.to(torch.device("cuda"))
            self.available = True
            print("✓ VAD model loaded")
        except Exception as e:
            print(f"✗ VAD unavailable: {e}")
            self.available = False
    
    def get_speech_segments(self, audio_path: str) -> List[Tuple[float, float]]:
        if not self.available:
            duration = librosa.get_duration(filename=audio_path)
            return [(0.0, duration)]
        
        try:
            diarization = self.pipeline(audio_path)
            segments = []
            
            for segment, _, label in diarization.itertracks(yield_label=True):
                if segment.start is not None and segment.end is not None:
                    segments.append((float(segment.start), float(segment.end)))
            
            return self._merge_segments(segments)
        except Exception as e:
            print(f"VAD failed: {e}")
            duration = librosa.get_duration(filename=audio_path)
            return [(0.0, duration)]
    
    @staticmethod
    def _merge_segments(segments, gap=0.5):
        if not segments:
            return segments
        
        segments = sorted(segments)
        merged = [segments[0]]
        
        for start, end in segments[1:]:
            if start - merged[-1][1] < gap:
                merged[-1] = (merged[-1][0], max(merged[-1][1], end))
            else:
                merged.append((start, end))
        
        return merged

# Smart Segmenter
class SmartAudioSegmenter:
    def __init__(self, output_dir: Path, min_duration: float = 30.0, use_vad: bool = True):
        self.output_dir = Path(output_dir) / "audio_segments"
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.min_duration = min_duration
        self.vad = VADProcessor() if use_vad else None
    
    def segment_file(self, audio_path: str, segments: List[WordSegment]) -> List[AudioChunk]:
        try:
            audio, sr = librosa.load(audio_path, sr=16000)
        except Exception as e:
            print(f"Failed to load {audio_path}: {e}")
            return []
        
        # Get VAD segments
        vad_segs = self.vad.get_speech_segments(audio_path) if self.vad else [(0, len(audio)/sr)]
        
        # Merge to minimum duration
        merged_groups = self._merge_to_minimum(segments, audio, sr, vad_segs)
        
        # Save chunks
        chunks = []
        for i, (group, start_time, end_time) in enumerate(merged_groups, 1):
            start_sample = int(start_time * sr)
            end_sample = int(end_time * sr)
            audio_chunk = audio[start_sample:end_sample]
            
            duration = len(audio_chunk) / sr
            
            # Save audio
            base_name = Path(audio_path).stem
            audio_file = self.output_dir / f"{base_name}_seg{i}.wav"
            sf.write(audio_file, audio_chunk, sr)
            
            # Save aligned CHA
            cha_file = self.output_dir / f"{base_name}_seg{i}.cha"
            self._save_aligned_cha(cha_file, group)
            
            chunks.append(AudioChunk(
                file_name=base_name,
                segment_number=i,
                audio_data=audio_chunk,
                sample_rate=sr,
                start_time=start_time,
                end_time=end_time,
                duration=duration,
                utterances=group
            ))
        
        return chunks
    
    def _merge_to_minimum(self, segments, audio, sr, vad_segs):
        """Merge segments to meet minimum duration"""
        merged = []
        current_group = []
        current_start = vad_segs[0][0] if vad_segs else 0
        current_end = vad_segs[0][1] if vad_segs else len(audio) / sr
        
        for seg_idx, (vad_start, vad_end) in enumerate(vad_segs):
            # Get segments in this VAD region
            region_segs = []
            for seg in segments:
                if seg.words:
                    seg_start = min(w[1] for w in seg.words) / 1000.0
                    seg_end = max(w[2] for w in seg.words) / 1000.0
                    
                    if seg_end >= vad_start and seg_start <= vad_end:
                        region_segs.append(seg)
            
            current_group.extend(region_segs)
            current_end = vad_end
            
            # Check if should finalize
            if (current_end - current_start) >= self.min_duration or seg_idx == len(vad_segs) - 1:
                if current_group:
                    merged.append((current_group, current_start, current_end))
                current_group = []
                current_start = vad_end
        
        return merged
    
    @staticmethod
    def _save_aligned_cha(output_path, segments):
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write("@Participants: CHILD\n")
            f.write(f"@Date: {datetime.now().strftime('%Y-%m-%d')}\n")
            f.write("@Language: fre\n\n")
            
            for seg in segments:
                f.write(f"*{seg.speaker}\t{seg.text}\n")
                f.write("\n")

print("✓ Smart segmenter loaded")

## ZONE 3: Segment Audio

In [ ]:
print("\n" + "="*70)
print("ZONE 3: SEGMENT AUDIO WITH VAD & MERGING")
print("="*70)

segmenter = SmartAudioSegmenter(
    CONFIG["output_dir"],
    min_duration=CONFIG["min_chunk_duration"],
    use_vad=CONFIG["use_vad"]
)

all_chunks = []

for cha_file, audio_file in match_result["matched"]:
    file_segments = [s for s in selected_segments if s.file_name == cha_file.stem]
    
    if file_segments:
        chunks = segmenter.segment_file(str(audio_file), file_segments)
        all_chunks.extend(chunks)
        print(f"  {cha_file.stem:25s}: {len(chunks):2d} chunks (≥{CONFIG['min_chunk_duration']}s each)")

print(f"\nTotal chunks created: {len(all_chunks)}")
print("="*70)

# ZONE 4: SPEAKER INFORMATION

In [ ]:
print("\n" + "="*70)
print("ZONE 4: SPEAKER INFORMATION")
print("="*70)

speakers_info = {}

for cha_file, _ in match_result["matched"]:
    with cha_file.open(encoding="utf-8") as f:
        content = f.read()
        match = re.search(r"@Participants:\s*(.+)", content)
        if match:
            participants = match.group(1)
            speakers_info[cha_file.stem] = participants
            print(f"  {cha_file.stem:25s}: {participants}")

print("="*70)

# ZONE 5: EVALUATE WITH WHISPER

In [ ]:
class WhisperEvaluator:
    def __init__(self, model_name: str = "base"):
        self.model = whisper.load_model(model_name)
    
    def evaluate_chunk(self, chunk: AudioChunk) -> Dict:
        temp_path = "/tmp/temp_audio.wav"
        sf.write(temp_path, chunk.audio_data, chunk.sample_rate)
        
        try:
            result = self.model.transcribe(temp_path, language="fr", verbose=False)
            hypothesis = result["text"].strip()
        except:
            hypothesis = ""
        
        reference = " ".join(seg.text for seg in chunk.utterances)
        wer_score = compute_wer(reference, hypothesis) if reference and hypothesis else 1.0
        
        return {
            "chunk_id": f"{chunk.file_name}_seg{chunk.segment_number}",
            "reference": reference,
            "hypothesis": hypothesis,
            "wer": wer_score,
            "duration": chunk.duration
        }
    
    def evaluate_chunks(self, chunks: List[AudioChunk], sample_size: Optional[int] = None) -> List[Dict]:
        if sample_size:
            chunks = chunks[:sample_size]
        
        results = []
        for chunk in tqdm(chunks, desc="Evaluating"):
            result = self.evaluate_chunk(chunk)
            results.append(result)
        
        return results

print("\n" + "="*70)
print("ZONE 5: EVALUATE WITH WHISPER")
print("="*70)

evaluator = WhisperEvaluator(CONFIG["whisper_model"])
results = evaluator.evaluate_chunks(all_chunks, CONFIG["sample_size"])

avg_wer = np.mean([r['wer'] for r in results])
print(f"\nAverage WER: {avg_wer:.4f}")
print("="*70)

# ZONE 6: CREATE TRAINING DATASET

In [ ]:
class DatasetBuilder:
    def __init__(self, output_dir: Path):
        self.output_dir = Path(output_dir) / "training_dataset"
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def create_dataset(self, results: List[Dict], train_ratio: float = 0.8):
        successful = [r for r in results if r['wer'] < 1.0]
        
        n_train = int(len(successful) * train_ratio)
        train_results = successful[:n_train]
        eval_results = successful[n_train:]
        
        self._write_jsonl(train_results, self.output_dir / "train.jsonl")
        self._write_jsonl(eval_results, self.output_dir / "eval.jsonl")
        
        metadata = {
            "total_segments": len(successful),
            "train_segments": len(train_results),
            "eval_segments": len(eval_results),
            "train_ratio": train_ratio,
            "avg_wer_train": float(np.mean([r['wer'] for r in train_results])) if train_results else 0,
            "avg_wer_eval": float(np.mean([r['wer'] for r in eval_results])) if eval_results else 0,
            "created_at": datetime.now().isoformat()
        }
        
        with open(self.output_dir / "metadata.json", 'w') as f:
            json.dump(metadata, f, indent=2)
        
        return metadata
    
    @staticmethod
    def _write_jsonl(results, output_path):
        with open(output_path, 'w') as f:
            for result in results:
                record = {
                    "text": result["reference"],
                    "metadata": {"wer": result["wer"], "duration": result["duration"]}
                }
                f.write(json.dumps(record) + '\n')

print("\n" + "="*70)
print("ZONE 6: CREATE TRAINING DATASET")
print("="*70)

builder = DatasetBuilder(CONFIG["output_dir"])
metadata = builder.create_dataset(results, CONFIG["train_ratio"])

print(f"\n✓ Train: {metadata['train_segments']} segments (WER: {metadata['avg_wer_train']:.4f})")
print(f"✓ Eval: {metadata['eval_segments']} segments (WER: {metadata['avg_wer_eval']:.4f})")
print(f"✓ Output: {CONFIG['output_dir']}/training_dataset/")
print("="*70)

## FINAL SUMMARY

In [ ]:
print("\n" + "="*80)
print("PIPELINE COMPLETE")
print("="*80)

print(f"\nInput:")
print(f"  CHA files: {len(match_result['matched'])}")
print(f"  Segments extracted: {len(segments)}")
print(f"  Segments selected: {len(selected_segments)}")

print(f"\nProcessing:")
print(f"  Audio chunks created: {len(all_chunks)}")
print(f"  Chunks duration: ≥{CONFIG['min_chunk_duration']}s")
print(f"  VAD used: {CONFIG['use_vad']}")
print(f"  CHA files aligned: ✓")

print(f"\nOutput:")
print(f"  Audio segments: {CONFIG['output_dir']}/audio_segments/")
print(f"    - *.wav (audio chunks)")
print(f"    - *.cha (aligned transcripts)")
print(f"  Training dataset: {CONFIG['output_dir']}/training_dataset/")
print(f"    - train.jsonl")
print(f"    - eval.jsonl")
print(f"    - metadata.json")

print(f"\nResults:")
print(f"  Total segments: {metadata['total_segments']}")
print(f"  Train WER: {metadata['avg_wer_train']:.4f}")
print(f"  Eval WER: {metadata['avg_wer_eval']:.4f}")
print("\n" + "="*80)